# Notebook 0 — Data Preparation & Exploratory Data Analysis
### Indoor Air Pollution Detection via Ontology-driven Graph Neural Networks

This is the **foundation** of the pipeline. Everything downstream (ontology, graphs, GNN
autoencoders, evaluation) consumes what this notebook produces.

**What this notebook does**
1. Loads both sensor deployments — `laboratory.csv` and `one_room_apartement.csv`.
2. Cleans them identically (physically-impossible negatives, duplicates) and puts each on a
   **uniform 2-minute time grid** so windows are temporally consistent.
3. Labels the **three known anomaly events** by date/location.
4. Applies the **5-class semantic state** system (`Excellent → Dangerous`) and **danger flags**
   to every variable — the layer the ontology will attach to each observation.
5. Runs a short **EDA** that confirms each event's signature and the variable correlations.
6. Saves clean, portable **CSV** outputs + a single `config.json` (the source of truth for
   variable categories, ontology classes, thresholds, event windows, and window parameters).

**Two deployments, non-overlapping in time and location**

| File | Period | Room | Event it carries | Signature |
|---|---|---|---|---|
| `laboratory.csv` | 2023-03-22 → 2023-06-06 | Lab | Apr 17 outage · May 21 fire | data gap · subtle (NO2↑, PM weak) |
| `one_room_apartement.csv` | 2023-06-24 → 2023-07-11 | Apartment | Jul 9 humidity/dust error | strong PM2.5 spike (artifact) |

Because the rooms differ, "normal" is defined **per deployment**; the ontology and graph
topology are shared. This is *detection*, not forecasting — nothing here predicts a future value.


In [ ]:
%matplotlib inline
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
sns.set_style("whitegrid")
print("Environment ready. pandas", pd.__version__, "| numpy", np.__version__)

## 1. Configuration — the single source of truth

Everything the rest of the project needs is defined here and exported to `config.json`:
the variable list, each variable's **ontology category + class**, units, which variables may be
floored at zero, the **5-class state thresholds**, the **event windows**, and the **window**
parameters. Downstream notebooks import this file instead of re-defining anything.

**Paths auto-detect**: the notebook looks for the two CSVs in `data/`, the current folder, then
the upload area. Put the two files in a `data/` folder next to this notebook and it just works.


In [ ]:
# --- paths (auto-detect the two CSVs) ---
_CANDIDATES = [Path("data"), Path("."), Path("/mnt/user-data/uploads")]
DATA_DIR = next((p for p in _CANDIDATES if (p / "laboratory.csv").exists()), Path("data"))
OUTPUT_DIR = Path("processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LAB_FILE = DATA_DIR / "laboratory.csv"
APT_FILE = DATA_DIR / "one_room_apartement.csv"
print("Reading from :", DATA_DIR.resolve())
print("Writing to   :", OUTPUT_DIR.resolve())

# --- resampling / windowing ---
FREQ = "2min"          # native cadence of both sensors
WINDOW_SIZE = 30       # samples per window  (~60 min)
WINDOW_STRIDE = 15     # step between windows (~30 min)
MAX_GAP_FRAC = 0.20    # drop windows with >20% missing samples

# --- ontology category per variable (broad class) ---
CATEGORY = {
    "co":"Pollutant","co2":"Pollutant","no2":"Pollutant","so2":"Pollutant","o3":"Pollutant","tvoc":"Pollutant",
    "pm1":"ParticleMatter","pm2_5":"ParticleMatter","pm10":"ParticleMatter","TypPS":"ParticleMatter",
    "cnt0_3":"ParticleCount","cnt0_5":"ParticleCount","cnt1":"ParticleCount",
    "cnt2_5":"ParticleCount","cnt5":"ParticleCount","cnt10":"ParticleCount",
    "temperature":"EnvironmentalCondition","temperature_o2":"EnvironmentalCondition",
    "humidity":"EnvironmentalCondition","humidity_abs":"EnvironmentalCondition",
    "dewpt":"EnvironmentalCondition","pressure":"EnvironmentalCondition","oxygen":"EnvironmentalCondition",
    "dCO2dt":"DynamicCondition","dHdt":"DynamicCondition",
    "sound":"AcousticCondition","sound_max":"AcousticCondition",
    "health":"AssessmentMetric","performance":"AssessmentMetric",
    "measuretime":"MeasurementProperty",
}

# --- specific ontology class per variable ---
ONTO_CLASS = {
    "co":"CarbonMonoxide","co2":"CarbonDioxide","no2":"NitrogenDioxide","so2":"SulfurDioxide",
    "o3":"Ozone","tvoc":"VOC","oxygen":"OxygenLevel",
    "pm1":"PM1","pm2_5":"PM2_5","pm10":"PM10","TypPS":"AverageParticleSize",
    "cnt0_3":"ParticleCount0_3","cnt0_5":"ParticleCount0_5","cnt1":"ParticleCount1",
    "cnt2_5":"ParticleCount2_5","cnt5":"ParticleCount5","cnt10":"ParticleCount10",
    "temperature":"Temperature","temperature_o2":"OxygenTemperature","humidity":"RelativeHumidity",
    "humidity_abs":"AbsoluteHumidity","dewpt":"DewPoint","pressure":"AtmosphericPressure",
    "dCO2dt":"CO2ChangeRate","dHdt":"HumidityChangeRate",
    "sound":"NoiseLevel","sound_max":"MaximumNoiseLevel",
    "health":"HealthIndex","performance":"PerformanceIndex","measuretime":"MeasurementCycle",
}

# --- units (from the dataset description) ---
UNITS = {
    "TypPS":"um","oxygen":"%vol","pm10":"ug/m3","cnt0_5":"count","co":"ppm","temperature":"C",
    "performance":"score","co2":"ppm","measuretime":"ms","so2":"ug/m3","no2":"ug/m3","cnt5":"count",
    "pm1":"ug/m3","cnt1":"count","dewpt":"C","tvoc":"ppb","pressure":"hPa","cnt10":"count",
    "dCO2dt":"ppm/s","sound_max":"dB(A)","health":"index","temperature_o2":"C","cnt2_5":"count",
    "o3":"ug/m3","humidity":"%","dHdt":"g/m3/s","humidity_abs":"g/m3","sound":"dB(A)","pm2_5":"ug/m3","cnt0_3":"count",
}

# Variables whose negatives are physically impossible -> floor at 0.
# (Concentrations, particle masses/counts, %, pressure, sound, performance.)
NONNEG_VARS = ["co","co2","no2","so2","o3","tvoc","oxygen","pm1","pm2_5","pm10","TypPS",
               "cnt0_3","cnt0_5","cnt1","cnt2_5","cnt5","cnt10",
               "humidity","humidity_abs","pressure","sound","sound_max","performance"]
# Variables where a negative is meaningful -> leave untouched.
SIGNED_VARS = ["temperature","temperature_o2","dewpt","dCO2dt","dHdt","health"]
RATE_VARS = ["dCO2dt","dHdt"]
META_VARS = ["measuretime"]   # device metadata, timestamp handled as index
print("Configured", len(CATEGORY), "variables across", len(set(CATEGORY.values())), "ontology categories.")

### 1b. The 5-class state thresholds

Three kinds of variable behaviour:

- **`asc`** — higher is worse (pollutants, particles, counts, noise, dew point). Four ascending
  cut-points map to `Excellent < Good < Moderate < Poor < Dangerous`.
- **`band`** — there is an *optimal range* and deviation in either direction is worse
  (temperature, humidity, pressure, oxygen). Nested bands, innermost = best.
- **`desc`** — higher is better (`performance`, `health`).

**Two corrections to the raw threshold table, made deliberately (and worth stating at the defense):**

1. **`health`** — the source table treated `health` like an AQI (low = good). The dataset
   description and the data itself (mean ≈ 831 on a 0–1000 scale) show the opposite: **higher =
   healthier**. We therefore classify `health` as `desc`, aligned to `performance`, and treat the
   special alarm codes `-200` (gas) / `-800` (fire) as `Dangerous`.
2. **`dCO2dt` / `dHdt`** — the table bands these at `<0.1 … >5`, but the actual rates swing in the
   ±100s, so those fixed bands are not meaningful. We instead fit their cut-points from the
   **data's own quantiles** (on the absolute rate), and flag this as data-driven.


In [ ]:
STATE_LEVELS = ["Excellent", "Good", "Moderate", "Poor", "Dangerous"]
DANGER_STATES = {"Poor", "Dangerous"}   # what raises hasDangerFlag = True

# kind, parameters.  asc -> 4 ascending cuts; desc -> 4 descending cuts; band -> nested ranges.
STATE_SPEC = {
    "co":        ("asc", [8, 15, 50, 200]),
    "co2":       ("asc", [800, 1000, 2000, 5000]),
    "tvoc":      ("asc", [300, 500, 1000, 3000]),
    "no2":       ("asc", [40, 80, 180, 280]),
    "so2":       ("asc", [40, 80, 380, 800]),
    "o3":        ("asc", [50, 100, 168, 208]),
    "pm1":       ("asc", [15, 35, 55, 150]),
    "pm2_5":     ("asc", [30, 60, 90, 120]),
    "pm10":      ("asc", [50, 100, 250, 350]),
    "TypPS":     ("asc", [0.3, 1, 2.5, 5]),
    "cnt0_3":    ("asc", [1000, 10000, 50000, 100000]),
    "cnt0_5":    ("asc", [1000, 10000, 50000, 100000]),
    "cnt1":      ("asc", [500, 5000, 20000, 50000]),
    "cnt2_5":    ("asc", [100, 1000, 5000, 10000]),
    "cnt5":      ("asc", [50, 500, 2000, 5000]),
    "cnt10":     ("asc", [20, 200, 1000, 3000]),
    "dewpt":     ("asc", [15, 18, 21, 24]),
    "sound":     ("asc", [40, 55, 70, 85]),
    "sound_max": ("asc", [45, 60, 80, 100]),
    "temperature":    ("band", {"Excellent":(21,25),"Good":(20,30),"Moderate":(15,35),"Poor":(12,40)}),
    "temperature_o2": ("band", {"Excellent":(21,25),"Good":(20,30),"Moderate":(15,35),"Poor":(12,40)}),
    "humidity":       ("band", {"Excellent":(40,50),"Good":(30,60),"Moderate":(20,70),"Poor":(10,80)}),
    "humidity_abs":   ("band", {"Excellent":(5,10),"Good":(3,12),"Moderate":(2,15),"Poor":(1,20)}),
    "pressure":       ("band", {"Excellent":(1000,1025),"Good":(980,1040),"Moderate":(950,1050),"Poor":(900,1050)}),
    "oxygen":         ("band", {"Excellent":(20.5,23),"Good":(19.5,23.5),"Moderate":(15,23.5),"Poor":(10,23.5)}),
    "performance": ("desc", [900, 700, 500, 300]),
    "health":      ("desc_alarm", [900, 700, 500, 300]),
    "dCO2dt":      ("asc_q", None),   # cuts fitted from data below
    "dHdt":        ("asc_q", None),   # cuts fitted from data below
    "measuretime": ("meta", None),
}

def _classify_asc(s, cuts):
    c = [s < cuts[0], s < cuts[1], s < cuts[2], s < cuts[3]]
    return pd.Series(np.select(c, STATE_LEVELS[:4], default="Dangerous"), index=s.index, dtype=object)

def _classify_desc(s, cuts):
    c = [s >= cuts[0], s >= cuts[1], s >= cuts[2], s >= cuts[3]]
    return pd.Series(np.select(c, STATE_LEVELS[:4], default="Dangerous"), index=s.index, dtype=object)

def _classify_band(s, bands):
    inb = lambda lo, hi: (s >= lo) & (s <= hi)
    c = [inb(*bands["Excellent"]), inb(*bands["Good"]), inb(*bands["Moderate"]), inb(*bands["Poor"])]
    return pd.Series(np.select(c, STATE_LEVELS[:4], default="Dangerous"), index=s.index, dtype=object)

def classify(series, kind, param):
    s = pd.to_numeric(series, errors="coerce")
    if kind == "meta":
        return pd.Series(np.nan, index=s.index, dtype=object)
    if kind == "asc":
        out = _classify_asc(s, param)
    elif kind == "asc_q":
        out = _classify_asc(s.abs(), param)            # magnitude of change
    elif kind == "desc":
        out = _classify_desc(s, param)
    elif kind == "desc_alarm":
        out = _classify_desc(s, param)
        out[s <= -100] = "Dangerous"                   # -200 gas / -800 fire codes
    elif kind == "band":
        out = _classify_band(s, param)
    else:
        raise ValueError(kind)
    return out.mask(s.isna())                          # gap rows -> no state
print("State system defined:", STATE_LEVELS)

### 1c. The three anomaly events

Each event lives in exactly one deployment. The power outage is **structural** (a data gap, not
anomalous readings) and is treated as a data-availability case rather than a reconstruction
target; the other two are **value** anomalies the GNN must catch.


In [ ]:
EVENTS = [
    {"name":"power_outage", "location":"laboratory", "ontology_class":"PowerOutage",
     "start":"2023-04-17 00:00", "end":"2023-04-19 12:00", "detection":"structural",
     "description":"Lab power outage (probable short circuit) -> appears as a ~44h data gap."},
    {"name":"fire", "location":"laboratory", "ontology_class":"FireEvent",
     "start":"2023-05-21 00:00", "end":"2023-05-21 23:59", "detection":"value",
     "description":"Large fire in Herzogenrath ~9-10 km away -> subtle NO2 rise, weak PM response."},
    {"name":"dust_humidity", "location":"apartment", "ontology_class":"HumidityMeasurementError",
     "start":"2023-07-09 00:00", "end":"2023-07-09 23:59", "detection":"value",
     "description":"Fine-dust measurement error from a sudden humidity increase -> strong PM2.5 artifact."},
]
for e in EVENTS:
    print(f"{e['name']:14s} | {e['location']:10s} | {e['detection']:10s} | {e['start']} -> {e['end']}")

## 2. Load both deployments

In [ ]:
def load_raw(path, location):
    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").drop_duplicates(subset="timestamp", keep="first")
    df["location"] = location
    return df.reset_index(drop=True)

lab_raw = load_raw(LAB_FILE, "laboratory")
apt_raw = load_raw(APT_FILE, "apartment")

VARS = [c for c in lab_raw.columns if c not in ("timestamp", "location")]   # 30 numeric variables
assert set(lab_raw.columns) == set(apt_raw.columns), "column mismatch between files"

for name, df in [("laboratory", lab_raw), ("apartment", apt_raw)]:
    print(f"{name:11s}: {df.shape[0]:6d} rows | {df['timestamp'].min()} -> {df['timestamp'].max()} "
          f"| {(df['timestamp'].max()-df['timestamp'].min()).days} days")
print("\nVariables (", len(VARS), "):", VARS)

## 3. Cleaning + uniform time grid

**Floor impossible negatives** on the non-negative variables (the electrochemical gas sensors
drift below zero — e.g. SO2 dips to ~ -163 — and optical counts can read slightly negative);
signed variables (temperatures, dew point, the rate terms, and `health`'s alarm codes) are left
untouched. Then each deployment is **resampled onto a strict 2-minute grid** with the mean per
bin. Empty bins become `NaN` rows and are tracked with an `is_gap` flag — this is exactly how the
April outage surfaces.


In [ ]:
def clean_and_regularize(df, location, freq=FREQ):
    df = df.copy()
    # 1) floor physically-impossible negatives
    for v in NONNEG_VARS:
        df[v] = df[v].clip(lower=0)
    # 2) regular time grid (mean per bin); empty bins -> NaN
    num = df.drop(columns=["location"]).set_index("timestamp")
    reg = num.resample(freq).mean()
    reg.index.name = "timestamp"
    # 3) gap flag + re-tag location
    reg["is_gap"] = reg[VARS].isna().all(axis=1)
    reg["location"] = location
    return reg

lab = clean_and_regularize(lab_raw, "laboratory")
apt = clean_and_regularize(apt_raw, "apartment")

for name, df in [("laboratory", lab), ("apartment", apt)]:
    g = int(df["is_gap"].sum())
    print(f"{name:11s}: {len(df):6d} grid rows | gap rows: {g:5d} ({100*g/len(df):4.1f}%) "
          f"| longest gap: {df['is_gap'].astype(int).groupby((~df['is_gap']).cumsum()).sum().max()} steps")

## 4. Label the anomaly events

In [ ]:
def label_events(df, location):
    df = df.copy()
    df["event"] = "normal"
    for e in EVENTS:
        if e["location"] != location:
            continue
        m = (df.index >= pd.Timestamp(e["start"])) & (df.index <= pd.Timestamp(e["end"]))
        df.loc[m, "event"] = e["name"]
    return df

lab = label_events(lab, "laboratory")
apt = label_events(apt, "apartment")

print("Event row counts:")
for name, df in [("laboratory", lab), ("apartment", apt)]:
    vc = df["event"].value_counts()
    print(f"  {name}: " + ", ".join(f"{k}={v}" for k, v in vc.items()))

## 5. Apply the 5-class states + danger flags

For each deployment we build a parallel **state table** (one column per variable holding
`Excellent … Dangerous`) and a per-row `n_danger` count (how many variables sit in
`Poor`/`Dangerous`). This is the semantic layer the ontology will attach via `hasState` /
`hasDangerFlag`.


In [ ]:
# fit data-driven cut-points for the rate variables (pooled across both deployments)
RATE_CUTS = {}
for v in RATE_VARS:
    pooled = pd.concat([lab[v].abs(), apt[v].abs()]).dropna()
    RATE_CUTS[v] = [round(float(x), 4) for x in pooled.quantile([0.50, 0.80, 0.95, 0.99]).values]
    print(f"{v}: data-driven |rate| cuts (q50/80/95/99) = {RATE_CUTS[v]}")

# resolve the runtime spec (inject the fitted rate cuts)
STATE_SPEC_RT = {}
for v, (kind, param) in STATE_SPEC.items():
    STATE_SPEC_RT[v] = (kind, RATE_CUTS[v] if kind == "asc_q" else param)

def build_states(df):
    states = pd.DataFrame(index=df.index)
    for v in VARS:
        kind, param = STATE_SPEC_RT.get(v, ("meta", None))
        states[v] = classify(df[v], kind, param)
    danger = states[VARS].isin(DANGER_STATES)
    states["n_danger"] = danger.sum(axis=1).where(~df["is_gap"])
    states["event"] = df["event"].values
    states["location"] = df["location"].values
    return states

lab_states = build_states(lab)
apt_states = build_states(apt)
print("\nSample (apartment, around the July 9 spike):")
cols = ["pm2_5", "pm10", "humidity", "no2", "co2", "n_danger"]
display(apt_states.loc["2023-07-09 12:00":"2023-07-09 12:20", cols])

## 6. EDA — do the three events actually look the way we claimed?

### 6a. July 9 (apartment) — humidity-induced fine-dust artifact
PM2.5 should rocket while humidity climbs. This is the easy, unambiguous detection.


In [ ]:
sub = apt.loc["2023-07-08":"2023-07-10"]
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(sub.index, sub["pm2_5"], color="tab:red", lw=1.1, label="PM2.5")
ax1.set_ylabel("PM2.5 (ug/m3)", color="tab:red"); ax1.tick_params(axis="y", labelcolor="tab:red")
ax2 = ax1.twinx()
ax2.plot(sub.index, sub["humidity"], color="tab:blue", lw=1.1, alpha=0.8, label="Humidity")
ax2.set_ylabel("Humidity (%)", color="tab:blue"); ax2.tick_params(axis="y", labelcolor="tab:blue")
ax1.set_title("Apartment — July 9 humidity-induced fine-dust measurement error")
ax1.set_xlabel("time"); fig.tight_layout(); plt.show()
print("July 9 PM2.5  max = %.1f ug/m3  (normal-day mean ~16)" % sub.loc["2023-07-09","pm2_5"].max())
print("July 9 humidity max = %.1f %%" % sub.loc["2023-07-09","humidity"].max())

### 6b. May 21 (lab) — distant fire
The hard case. NO2 lifts a little; PM barely reacts (the fire was ~10 km away). The point of
showing this is honesty about how *subtle* the signal is — the model is not expected to catch
this as easily as July 9.


In [ ]:
sub = lab.loc["2023-05-20":"2023-05-22"]
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(sub.index, sub["no2"], color="tab:purple", lw=1.1, label="NO2")
ax1.set_ylabel("NO2 (ug/m3)", color="tab:purple"); ax1.tick_params(axis="y", labelcolor="tab:purple")
ax2 = ax1.twinx()
ax2.plot(sub.index, sub["pm2_5"], color="tab:red", lw=1.0, alpha=0.7, label="PM2.5")
ax2.set_ylabel("PM2.5 (ug/m3)", color="tab:red"); ax2.tick_params(axis="y", labelcolor="tab:red")
ax1.set_title("Laboratory — May 21 distant-fire signature (subtle: NO2 up, PM weak)")
ax1.set_xlabel("time"); fig.tight_layout(); plt.show()
fire = lab.loc["2023-05-21"]; norm = lab.loc["2023-05-14"]
print("NO2  mean  fire-day=%.1f  vs normal=%.1f" % (fire["no2"].mean(), norm["no2"].mean()))
print("PM2.5 mean fire-day=%.1f  vs normal=%.1f" % (fire["pm2_5"].mean(), norm["pm2_5"].mean()))

### 6c. April 17–19 (lab) — power outage = a structural gap
No values to reconstruct here; the anomaly *is* the absence of data. Daily valid-sample counts
make the outage obvious.


In [ ]:
daily = (~lab["is_gap"]).resample("1D").sum()
apr = daily.loc["2023-04-10":"2023-04-25"]
fig, ax = plt.subplots(figsize=(11, 3.6))
colors = ["tab:red" if v < 200 else "tab:blue" for v in apr.values]
ax.bar(apr.index, apr.values, width=0.85, color=colors)
ax.axhline(720, color="grey", ls="--", lw=0.8, label="expected full day (~720)")
ax.set_title("Laboratory — daily valid-sample count (April): the Apr 17-19 power-outage gap in red")
ax.set_ylabel("samples / day"); ax.legend(); fig.tight_layout(); plt.show()

### 6d. Variable correlations (both deployments)
Confirms the structure that justifies ontology edges: PM1/PM2.5/PM10 move together, humidity
relates to dew point, etc. This echoes the heatmap analysis in the earlier (RNN/LSTM/GRU) work
and informs the optional correlation edges in Notebook 2.


In [ ]:
core = ["co2","co","no2","so2","o3","tvoc","pm1","pm2_5","pm10",
        "temperature","humidity","dewpt","pressure","oxygen","sound"]
pooled = pd.concat([lab[core], apt[core]]).dropna()
corr = pooled.corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            square=True, cbar_kws={"shrink": 0.8}, annot_kws={"size": 7}, ax=ax)
ax.set_title("Correlation across both deployments (core variables)")
fig.tight_layout(); plt.show()

### 6e. State distribution per variable (reality check)
Most gases and particulates sit in `Excellent`/`Good` almost all the time. That is expected and
*useful*: it gives the autoencoder a clean, well-defined notion of "normal" to learn.


In [ ]:
show = ["co2","tvoc","no2","so2","o3","pm2_5","pm10","temperature","humidity","dewpt","sound"]
allstates = pd.concat([lab_states[show], apt_states[show]])
prop = pd.DataFrame({v: allstates[v].value_counts(normalize=True).reindex(STATE_LEVELS).fillna(0)
                     for v in show}).T
palette = {"Excellent":"#2ecc71","Good":"#a3d977","Moderate":"#f1c40f","Poor":"#e67e22","Dangerous":"#e74c3c"}
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(show))
for st in STATE_LEVELS:
    ax.bar(prop.index, prop[st].values, bottom=bottom, color=palette[st], label=st)
    bottom += prop[st].values
ax.set_ylabel("proportion of observations")
ax.set_title("State distribution per variable (both deployments)")
ax.legend(ncol=5, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.04))
plt.xticks(rotation=45, ha="right"); fig.tight_layout(); plt.show()

## 7. Window manifest (light)

The GNN works on **windows** of consecutive samples. Here we only build the *manifest* — window
id, location, time span, missing fraction, and a label (majority non-normal event in the window,
else `normal`). Turning each usable window into a graph (node features on the shared topology)
is Notebook 2's job; keeping it separate keeps window-size a single tunable knob.


In [ ]:
def make_windows(df, name, size=WINDOW_SIZE, stride=WINDOW_STRIDE, max_gap=MAX_GAP_FRAC):
    gap = df[VARS].isna().all(axis=1).values
    ev = df["event"].values
    idx = df.index
    rows, wid = [], 0
    for s in range(0, len(df) - size + 1, stride):
        e = s + size
        seg = ev[s:e]
        nonnorm = [x for x in seg if x != "normal"]
        label = max(set(nonnorm), key=nonnorm.count) if nonnorm else "normal"
        gfrac = float(gap[s:e].mean())
        rows.append({"window_id": f"{name}_{wid:05d}", "location": name,
                     "start": idx[s], "end": idx[e - 1], "n_samples": size,
                     "gap_frac": round(gfrac, 3), "event": label, "usable": gfrac <= max_gap})
        wid += 1
    return pd.DataFrame(rows)

windows = pd.concat([make_windows(lab, "laboratory"), make_windows(apt, "apartment")],
                    ignore_index=True)
print("Total windows:", len(windows), "| usable:", int(windows["usable"].sum()))
print("\nUsable windows by event:")
print(windows[windows["usable"]].groupby(["location", "event"]).size().to_string())

## 8. Save outputs

Portable CSVs (no special reader needed) plus one `config.json`. The timestamp is written as a
column so any tool can re-load it.


In [ ]:
def save_csv(df, path):
    df.reset_index().to_csv(path, index=False)

save_csv(lab, OUTPUT_DIR / "laboratory_clean.csv")
save_csv(apt, OUTPUT_DIR / "apartment_clean.csv")
save_csv(lab_states, OUTPUT_DIR / "laboratory_states.csv")
save_csv(apt_states, OUTPUT_DIR / "apartment_states.csv")
windows.to_csv(OUTPUT_DIR / "windows.csv", index=False)

# JSON-friendly state spec (tuples -> lists)
spec_json = {}
for v, (kind, param) in STATE_SPEC_RT.items():
    if isinstance(param, dict):
        param = {k: list(val) for k, val in param.items()}
    elif isinstance(param, (list, tuple)):
        param = [float(x) for x in param]
    spec_json[v] = {"kind": kind, "param": param}

config = {
    "variables": VARS,
    "category": CATEGORY,
    "ontology_class": ONTO_CLASS,
    "units": UNITS,
    "nonneg_vars": NONNEG_VARS,
    "signed_vars": SIGNED_VARS,
    "rate_vars": RATE_VARS,
    "meta_vars": META_VARS,
    "state_levels": STATE_LEVELS,
    "danger_states": sorted(DANGER_STATES),
    "state_spec": spec_json,
    "rate_cuts": RATE_CUTS,
    "events": EVENTS,
    "freq": FREQ,
    "window": {"size": WINDOW_SIZE, "stride": WINDOW_STRIDE, "max_gap_frac": MAX_GAP_FRAC},
    "deployments": {
        "laboratory": {"rows": int(len(lab)),
                       "start": str(lab.index.min()), "end": str(lab.index.max())},
        "apartment":  {"rows": int(len(apt)),
                       "start": str(apt.index.min()), "end": str(apt.index.max())},
    },
}
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved to", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {p.name:28s} {p.stat().st_size/1024:8.1f} KB")

## 9. Summary & hand-off to Notebook 1

**Produced**
- `laboratory_clean.csv`, `apartment_clean.csv` — cleaned, 2-min grid, `is_gap` + `event` columns.
- `laboratory_states.csv`, `apartment_states.csv` — 5-class state per variable + `n_danger`.
- `windows.csv` — window manifest (usable flag + event label) for the graph stage.
- `config.json` — variables, ontology categories/classes, units, state thresholds, event
  windows, window parameters. **The single source of truth** every later notebook imports.

**Confirmed in EDA**
- July 9 is a strong PM2.5 artifact (easy detection); May 21 fire is subtle (the hard case);
  April 17–19 is a structural gap (availability, not reconstruction).
- PM1/PM2.5/PM10 are tightly correlated; gases & PM sit mostly in `Excellent`/`Good` — a clean
  "normal" for the autoencoder.

**Next — Notebook 1: Ontology construction.** Read `config.json`, build the OWL/RDF ontology
(classes, object + data properties), generate observation individuals carrying `hasValue` /
`hasState` / `hasDangerFlag`, run reasoning, export OWL + RDF, and report coverage
(31/31 variables, triple counts, a single connected component).
